In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

#  MACHINE TRANSLATION PROJECT - ENGLISH TO INDIAN LANGUAGES

In [ ]:
import numpy as np
import pandas as pd
import os, json, re, string, math, time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
import nltk
nltk.download('punkt_tab')

# Install Indic NLP library
!pip install indic-nlp-library
from indicnlp.tokenize import indic_tokenize
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

# Install torchviz library
!pip install torchviz
from torchviz import make_dot

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")


#  CONFIGURATION

In [ ]:
class Config:
    SEQ_LENGTH = 50
    BATCH_SIZE = 128
    EMBEDDING_DIM = 256
    UNITS = 1024
    EPOCHS = 10
    LEARNING_RATE = 0.001
    TEACHER_FORCING_RATIO = 0.5
    GRAD_CLIP = 1.0
    BEAM_WIDTH = 5
    DROPOUT = 0.1

config = Config()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
train_path = "/kaggle/input/train_data1.json"
test_path = "/kaggle/input/test_data1_final.json"

#  NORMALIZERS & TOKENIZERS

In [ ]:
factory = IndicNormalizerFactory()
hi_normalizer = factory.get_normalizer("hi", remove_nuktas=False)
bn_normalizer = factory.get_normalizer("bn", remove_nuktas=False)

def preprocess_english(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        sentence = ''.join([ch for ch in sentence if ch not in string.punctuation or ch in ['.', '!', '?']])
        tokens = nltk.word_tokenize(sentence.lower())
        tokenized_sentences.append(tokens)
    return tokenized_sentences

def preprocess_hindi(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        normalized_sentence = hi_normalizer.normalize(sentence)
        tokens = indic_tokenize.trivial_tokenize(normalized_sentence)
        tokenized_sentences.append(tokens)
    return tokenized_sentences

def preprocess_bengali(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        normalized_sentence = bn_normalizer.normalize(sentence)
        tokens = indic_tokenize.trivial_tokenize(normalized_sentence)
        tokenized_sentences.append(tokens)
    return tokenized_sentences

#  VISUALIZATION FUNCTIONS

In [ ]:
# Create folder to save all visualization outputs
os.makedirs("visualizations", exist_ok=True)

def save_and_show(fig, filename, tight=True):
    """Helper to save figures with consistent format and quality"""
    filepath_png = os.path.join("visualizations", f"{filename}.png")
    if tight:
        fig.tight_layout()
    fig.savefig(filepath_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {filepath_png}")

In [ ]:
def plot_token_distribution(tokenized_corpus, title):
    all_tokens = [token for sent in tokenized_corpus for token in sent]
    counter = Counter(all_tokens)
    most_common = counter.most_common(20)
    words, freqs = zip(*most_common)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=list(freqs), y=list(words), ax=ax, palette="Blues_r")
    ax.set_title(f"Top 20 Most Frequent Tokens in {title}")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Token")
    save_and_show(fig, f"token_frequency_{title.lower().replace(' ', '_')}")

In [ ]:
def plot_sentence_lengths(corpus, lang_name):
    lengths = [len(sent) for sent in corpus]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(lengths, bins=40, color='skyblue', edgecolor='black')
    ax.set_title(f"Sentence Length Distribution ({lang_name})")
    ax.set_xlabel("Sentence Length (tokens)")
    ax.set_ylabel("Frequency")
    save_and_show(fig, f"sentence_length_{lang_name.lower()}")

In [ ]:
def plot_zipf_distribution(tokenized_corpus, lang_name):
    all_tokens = [token for sent in tokenized_corpus for token in sent]
    counter = Counter(all_tokens)
    frequencies = np.array(sorted(counter.values(), reverse=True))
    ranks = np.arange(1, len(frequencies) + 1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(ranks, frequencies, marker='.', linestyle='none', alpha=0.6)
    ax.set_title(f"Zipf's Law Distribution - {lang_name}")
    ax.set_xlabel("Rank (log scale)")
    ax.set_ylabel("Frequency (log scale)")
    ax.grid(True, which="both", ls="--", alpha=0.5)
    save_and_show(fig, f"zipf_distribution_{lang_name.lower().replace(' ', '_')}")

In [ ]:
def fit_zipf_slope(tokenized_corpus, lang_name):
    counter = Counter([token for sent in tokenized_corpus for token in sent])
    freqs = np.array(sorted(counter.values(), reverse=True))
    ranks = np.arange(1, len(freqs) + 1)
    log_ranks = np.log(ranks)
    log_freqs = np.log(freqs)
    slope, intercept, r_value, _, _ = linregress(log_ranks, log_freqs)
    print(f"{lang_name} — Zipf Slope (α): {abs(slope):.2f}, R² = {r_value**2:.4f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(log_ranks, log_freqs, '.', alpha=0.6, label=f"Data (α={abs(slope):.2f})")
    ax.plot(log_ranks, intercept + slope * log_ranks, 'r--', label="Best Fit Line")
    ax.set_xlabel("log(Rank)")
    ax.set_ylabel("log(Frequency)")
    ax.set_title(f"Zipf Fit for {lang_name}")
    ax.legend()
    save_and_show(fig, f"zipf_fit_{lang_name.lower().replace(' ', '_')}")

#  MODEL ARCHITECTURE

In [ ]:
# Encoder with Bidirectional GRU
class Encoder(nn.Module):
    def __init__(self, vocab_size, EMBEDDING_DIM, enc_UNITS, batch_sz, dropout=0.1):
        super().__init__()
        self.batch_sz = batch_sz
        self.enc_UNITS = enc_UNITS
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM)
        self.gru = nn.GRU(EMBEDDING_DIM, enc_UNITS, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(enc_UNITS * 2, enc_UNITS)
    
    def forward(self, x, lens, device):
        x = self.embedding(x)
        x = self.dropout(x)
        x = pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        hidden = self.initialize_hidden_state(device)
        output, hidden = self.gru(x, hidden)
        output, _ = pad_packed_sequence(output, batch_first=True, total_length=128)
        output = output[:, :, :self.enc_UNITS] + output[:, :, self.enc_UNITS:]
        hidden_combined = torch.tanh(self.fc(torch.cat((hidden[0:1], hidden[1:2]), dim=2)))
        return output, hidden_combined
    
    def initialize_hidden_state(self, device):
        return torch.zeros((2, self.batch_sz, self.enc_UNITS), device=device)

# Enhanced Decoder with Cross Attention
class Decoder(nn.Module):
    def __init__(self, vocab_size, EMBEDDING_DIM, dec_UNITS, enc_UNITS, batch_sz, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(EMBEDDING_DIM + enc_UNITS, dec_UNITS, batch_first=True)
        self.fc = nn.Linear(dec_UNITS, vocab_size)
        self.W1 = nn.Linear(enc_UNITS, dec_UNITS)
        self.W2 = nn.Linear(dec_UNITS, dec_UNITS)
        self.V = nn.Linear(dec_UNITS, 1)
    
    def forward(self, x, hidden, enc_output):
        hidden_with_time_axis = hidden.permute(1, 0, 2)
        score = torch.tanh(self.W1(enc_output) + self.W2(hidden_with_time_axis))
        attention_weights = torch.softmax(self.V(score), dim=1)
        context_vector = torch.sum(attention_weights * enc_output, dim=1)
        x = self.embedding(x)
        x = self.dropout(x)
        x = torch.cat((context_vector.unsqueeze(1), x), -1)
        output, state = self.gru(x, hidden)
        output = self.fc(output.squeeze(1))
        return output, state, attention_weights

#  ATTENTION HEATMAP FUNCTIONS

In [ ]:
def plot_attention_heatmap(attention_matrix, input_tokens, output_tokens, filename):
    """
    Plot and save an attention heatmap between input and output tokens.
    """
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention_matrix, cmap='viridis', xticklabels=input_tokens, yticklabels=output_tokens)
    plt.xlabel("Source Tokens")
    plt.ylabel("Target Tokens")
    plt.title(f"Attention Heatmap: {filename.split('/')[-1].replace('_', ' ').replace('.png','')}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved attention heatmap: {filename}")

def generate_attention_heatmap(encoder, decoder, sentence, en_word2index, de_index2word, lang_code, device, max_len=50):
    """
    Generate attention weights for one input sentence and save a heatmap.
    """
    encoder.eval()
    decoder.eval()

    # Tokenize the input sentence
    tokens = nltk.word_tokenize(sentence.lower())
    encoded = encode_and_pad(en_word2index, tokens, max_len)
    input_tensor = torch.tensor(encoded, dtype=torch.long, device=device).unsqueeze(0)
    input_len = torch.tensor([len(tokens)], dtype=torch.long, device=device)

    # Encode
    with torch.no_grad():
        enc_output, enc_hidden = encoder(input_tensor, input_len, device)
    
    # Decode (greedy for simplicity)
    dec_hidden = enc_hidden
    dec_input = torch.tensor([[en_word2index["<SOS>"]]], device=device)
    
    decoded_tokens = []
    attentions = []

    for t in range(max_len):
        with torch.no_grad():
            predictions, dec_hidden, attention_weights = decoder(dec_input, dec_hidden, enc_output)
        
        attention_vector = attention_weights.squeeze(0).detach().cpu().numpy()
        attentions.append(attention_vector)
        
        pred_token = predictions.argmax(1).item()
        decoded_tokens.append(pred_token)

        if pred_token == en_word2index["<EOS>"]:
            break
        
        dec_input = torch.tensor([[pred_token]], device=device)
    
    # Convert tokens to readable form
    input_words = tokens
    output_words = [de_index2word[idx] for idx in decoded_tokens if idx not in [0, en_word2index["<SOS>"], en_word2index["<EOS>"]]]

    # Stack attention matrix (target_len × source_len)
    attention_matrix = np.vstack(attentions[:len(output_words)])

    # Save heatmap
    os.makedirs("visualizations", exist_ok=True)
    filename = f"visualizations/attention_heatmap_{lang_code}.png"
    plot_attention_heatmap(attention_matrix, input_words, output_words, filename)

#  HELPER FUNCTIONS AND CLASSES

In [ ]:
def encode_and_pad(vocab, sent, max_length):
    sos = [vocab["<SOS>"]]; eos = [vocab["<EOS>"]]; pad = [vocab["<PAD>"]]
    encoded = [vocab.get(w, vocab["<PAD>"]) for w in sent]
    if len(sent) < max_length - 2:
        n_pads = max_length - 2 - len(sent)
        return sos + encoded + eos + pad * n_pads
    else:
        truncated = encoded[:max_length - 2]
        return sos + truncated + eos

In [ ]:
def sort_batch(X, y, lengths):
    lengths, indx = lengths.sort(dim=0, descending=True)
    X = X[indx]
    y = y[indx]
    return X, y, lengths

In [ ]:
def loss_function(real, pred, criterion):
    mask = real.ge(1).type(torch.cuda.FloatTensor if torch.cuda.is_available() else torch.FloatTensor)
    loss_ = criterion(pred, real) * mask
    return torch.mean(loss_)

In [ ]:
# Perform beam search decoding for a single input sequence.
def beam_search_decode(encoder, decoder, input_sequence, max_len, beam_width, sos_idx, eos_idx, de_index2word, device):
    encoder.eval()
    decoder.eval()
    
    with torch.no_grad():
        # Encode the input sequence
        input_len = torch.tensor([(input_sequence != 0).sum().item()], device=device)
        enc_output, enc_hidden = encoder(input_sequence.unsqueeze(0), input_len, device)
        
        # Initialize beam: (sequence, score, hidden_state)
        # Start with SOS token
        beam = [([sos_idx], 0.0, enc_hidden)]
        completed = []
        
        for _ in range(max_len):
            new_beam = []
            
            for seq, score, hidden in beam:
                # If the sequence ended with EOS, add to completed
                if seq[-1] == eos_idx:
                    completed.append((seq, score))
                    continue
                
                # Prepare decoder input
                dec_input = torch.tensor([[seq[-1]]], device=device)
                
                # Get decoder output
                predictions, new_hidden, _ = decoder(dec_input, hidden, enc_output)
                log_probs = F.log_softmax(predictions, dim=-1)
                
                # Get top k candidates
                topk_probs, topk_indices = torch.topk(log_probs, beam_width, dim=1)
                
                for i in range(beam_width):
                    next_token = topk_indices[0, i].item()
                    next_score = score + topk_probs[0, i].item()
                    new_seq = seq + [next_token]
                    new_beam.append((new_seq, next_score, new_hidden))
            
            if not new_beam:
                break
                
            # Keep top beam_width sequences
            new_beam.sort(key=lambda x: x[1], reverse=True)
            beam = new_beam[:beam_width]
        
        # Add any remaining sequences in beam to completed
        completed.extend([(seq, score) for seq, score, _ in beam if seq[-1] == eos_idx])
        
        if not completed:
            # If no sequences completed, use the best from beam
            completed = [(seq, score) for seq, score, _ in beam[:1]]
        
        # Select the sequence with highest score
        best_seq, best_score = max(completed, key=lambda x: x[1])
        
        # Convert to words
        translation = [
            de_index2word[idx]
            for idx in best_seq
            if idx not in [sos_idx, eos_idx, 0]  # Exclude <SOS>, <EOS>, and <PAD>
        ]
        
        return " ".join(translation)


In [ ]:
# Translate the entire test dataset using beam search and save predictions to CSV.
def save_predictions_with_beam_search(encoder, decoder, test_dl, test_ids, max_len, sos_idx, eos_idx, de_index2word, output_file, device, beam_width=5):
    encoder.eval()
    decoder.eval()
    
    translations = []
    
    with torch.no_grad():
        for batch_idx, (inp,) in enumerate(test_dl):
            inp = inp.to(device)
            
            # Process each sequence in the batch individually for beam search
            for i in range(inp.size(0)):
                input_sequence = inp[i]
                translation = beam_search_decode(
                    encoder, decoder, input_sequence, max_len, beam_width, 
                    sos_idx, eos_idx, de_index2word, device
                )
                translations.append(translation)
                
            print(f"Processed batch {batch_idx + 1}/{len(test_dl)}")
    
    # Create DataFrame with test IDs and translations
    df = pd.DataFrame({"ID": test_ids[:len(translations)], "Translation": translations})
    df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")



In [ ]:
# Learning Rate Scheduler with Warmup
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, max_lr, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.current_step = 0
        
    def step(self):
        self.current_step += 1
        lr = self.get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
            
    def get_lr(self):
        if self.current_step < self.warmup_steps:
            # Linear warmup
            return self.max_lr * (self.current_step / self.warmup_steps)
        else:
            # Cosine decay
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
            return self.min_lr + (self.max_lr - self.min_lr) * cosine_decay

In [ ]:
# Enhanced Training Loop with all improvements
def train_model(encoder, decoder, train_dl, en_word2index, de_word2index, config, device):
    # Optimizer
    optimizer = optim.Adam(
        list(encoder.parameters()) + list(decoder.parameters()), 
        lr=config.LEARNING_RATE,
        weight_decay=1e-5
    )
    
    # Learning rate scheduler with warmup
    total_steps = len(train_dl) * config.EPOCHS
    warmup_steps = total_steps // 10  # 10% warmup
    scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_steps, config.LEARNING_RATE)
    
    # Loss function
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore <PAD> token
    
    # Training history
    train_losses = []
    
    for epoch in range(config.EPOCHS):
        start = time.time()
        encoder.train()
        decoder.train()
        
        total_loss = 0
        batch_count = 0
        
        for (batch, (inp, targ, inp_len)) in enumerate(train_dl):
            loss = 0
            
            # Move inputs to device
            inp, targ, inp_len = inp.to(device), targ.to(device), inp_len.to(device)
            
            # Sort batch by sequence lengths
            xs, ys, lens = sort_batch(inp, targ, inp_len)
            xs, ys, lens = xs.to(device), ys.to(device), lens.to(device)
            
            # Encoder forward pass
            enc_output, enc_hidden = encoder(xs, lens, device)
            dec_hidden = enc_hidden
            
            # Initialize decoder input with <SOS>
            dec_input = torch.tensor([[en_word2index["<SOS>"]]] * config.BATCH_SIZE, device=device)
            
            # Teacher forcing with probability
            use_teacher_forcing = True if np.random.random() < config.TEACHER_FORCING_RATIO else False
            
            if use_teacher_forcing:
                # Teacher forcing: feed the target as the next input
                for t in range(1, ys.size(1)):
                    predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
                    loss += loss_function(ys[:, t], predictions, criterion)
                    dec_input = ys[:, t].unsqueeze(1)  # Teacher forcing
            else:
                # Without teacher forcing: use own predictions
                for t in range(1, ys.size(1)):
                    predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
                    loss += loss_function(ys[:, t], predictions, criterion)
                    
                    # Get the most likely next token
                    top1 = predictions.argmax(1)
                    dec_input = top1.unsqueeze(1)  # Use prediction as next input
            
            # Normalize batch loss
            batch_loss = loss / ys.size(1)
            total_loss += batch_loss.item()
            batch_count += 1
            
            # Backpropagation
            optimizer.zero_grad()
            batch_loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()), 
                config.GRAD_CLIP
            )
            
            optimizer.step()
            scheduler.step()
            
            # Log progress
            if batch % 50 == 0:
                current_lr = scheduler.get_lr()
                print(f"Epoch {epoch + 1} Batch {batch} Loss {batch_loss.item():.4f} LR {current_lr:.6f}")
        
        # Calculate average loss for the epoch
        avg_loss = total_loss / batch_count
        train_losses.append(avg_loss)
        
        # Log epoch details
        print(f"Epoch {epoch + 1} Completed - Average Loss: {avg_loss:.4f}")
        print(f"Time taken: {time.time() - start:.2f} sec")
        print(f"Current Learning Rate: {scheduler.get_lr():.6f}\n")
    
    return train_losses


# ENGLISH-BENGALI TRANSLATION

## Data Loading

In [ ]:
# Process JSON data for English-Bengali
source_sentences_train = []
target_sentences_train = []
source_sentences_test = []
id_train = []
id_test = []

with open(train_path, 'r') as file:
    data = json.load(file)

for language_pair, language_data in data.items():
    if language_pair == "English-Bengali":
        print(f"Language Pair: {language_pair}")
        for data_type, data_entries in language_data.items():
            print(f"  Data Type: {data_type}")
            for entry_id, entry_data in data_entries.items():
                source = entry_data["source"]
                if data_type == "Test":
                    source_sentences_test.append(source)
                    id_test.append(entry_id)
                else:
                    target = entry_data["target"]
                    source_sentences_train.append(source)
                    target_sentences_train.append(target)
                    id_train.append(entry_id)

with open(test_path, 'r') as file:
    data = json.load(file)

for language_pair, language_data in data.items():
    if language_pair == "English-Bengali":
        print(f"Language Pair: {language_pair}")
        for data_type, data_entries in language_data.items():
            print(f"  Data Type: {data_type}")
            for entry_id, entry_data in data_entries.items():
                source = entry_data["source"]
                if data_type == "Test":
                    source_sentences_test.append(source)
                    id_test.append(entry_id)
                else:
                    target = entry_data["target"]
                    source_sentences_train.append(source)
                    target_sentences_train.append(target)
                    id_train.append(entry_id)

print(f"Training samples: {len(source_sentences_train)}")
print(f"Test samples: {len(source_sentences_test)}")

## Data Preprocessing

In [ ]:
target_sentences_train = [re.sub(r'[a-zA-Z]','',hi) for hi in target_sentences_train] #optional

english_tokens = preprocess_english(source_sentences_train)
english_test = preprocess_english(source_sentences_test)
bengali_tokens = preprocess_bengali(target_sentences_train)

en_train = english_tokens
en_test = english_test
de_train = bengali_tokens

##  VISUALIZE CORPORA (English - Bengali)

In [ ]:
plot_token_distribution(english_tokens, "English Corpus")
plot_token_distribution(bengali_tokens, "Bengali Corpus")

plot_sentence_lengths(english_tokens, "English")
plot_sentence_lengths(bengali_tokens, "Bengali")

plot_zipf_distribution(english_tokens, "English Corpus")
plot_zipf_distribution(bengali_tokens, "Bengali Corpus")

fit_zipf_slope(english_tokens, "English Corpus")
fit_zipf_slope(bengali_tokens, "Bengali Corpus")

## Build vocabulary

In [ ]:
en_index2word = ["<PAD>", "<SOS>", "<EOS>"]
de_index2word = ["<PAD>", "<SOS>", "<EOS>"]

for ds in [en_train, en_test]:
    for sent in ds:
        for token in sent:
            if token not in en_index2word:
                en_index2word.append(token)

for ds in [de_train]:
    for sent in ds:
        for token in sent:
            if token not in de_index2word:
                de_index2word.append(token)

en_word2index = {token: idx for idx, token in enumerate(en_index2word)}
de_word2index = {token: idx for idx, token in enumerate(de_index2word)}

print(f"English vocabulary size: {len(en_word2index)}")
print(f"Bengali vocabulary size: {len(de_word2index)}")

## Encode and pad sequences

In [ ]:
en_train_encoded = [encode_and_pad(en_word2index, sent, config.SEQ_LENGTH) for sent in en_train]
en_test_encoded = [encode_and_pad(en_word2index, sent, config.SEQ_LENGTH) for sent in en_test]
de_train_encoded = [encode_and_pad(de_word2index, sent, config.SEQ_LENGTH) for sent in de_train]

## Prepare DataLoader

In [ ]:
train_x = np.array(en_train_encoded)
train_y = np.array(de_train_encoded)
test_x = np.array(en_test_encoded)

train_lens = torch.tensor([len(seq) - seq.count(0) for seq in train_x.tolist()], dtype=torch.long)

train_ds = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y), train_lens)
train_dl = DataLoader(train_ds, shuffle=True, batch_size=config.BATCH_SIZE, drop_last=True)

test_ds = TensorDataset(torch.from_numpy(test_x))
test_dl = DataLoader(test_ds, batch_size=config.BATCH_SIZE)

print(f"Training DataLoader: {len(train_dl)} batches")
print(f"Test DataLoader: {len(test_dl)} batches")

## Initialize models

In [ ]:
encoder = Encoder(len(en_word2index), config.EMBEDDING_DIM, config.UNITS, config.BATCH_SIZE, config.DROPOUT).to(device)
decoder = Decoder(len(de_word2index), config.EMBEDDING_DIM, config.UNITS, config.UNITS, config.BATCH_SIZE, config.DROPOUT).to(device)

print("Models initialized successfully!")
print(f"Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Decoder parameters: {sum(p.numel() for p in decoder.parameters()):,}")

## Train the model

In [ ]:
print("Starting training...")
train_losses = train_model(encoder, decoder, train_dl, en_word2index, de_word2index, config, device)

## Plotting the training and validation loss curves

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have a list `train_losses` storing loss per epoch
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o')
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.savefig("training_loss_curve_1.png", dpi=300)
plt.close()

print("Saved training loss curve as training_loss_curve_1.png")


## Save predictions with beam search

In [ ]:
print("Generating predictions with beam search...")
save_predictions_with_beam_search(
    encoder=encoder,
    decoder=decoder,
    test_dl=test_dl,
    test_ids=id_test,
    max_len=config.SEQ_LENGTH,
    sos_idx=en_word2index["<SOS>"],
    eos_idx=en_word2index["<EOS>"],
    de_index2word=de_index2word,
    output_file="answersB.csv",
    device=device,
    beam_width=config.BEAM_WIDTH
)

print("Training and prediction completed successfully!")

## ATTENTION HEATMAP VISUALIZATION

In [ ]:
sample_sentence_bn = "The government announced new health policies."
# Generate Bengali attention heatmap
generate_attention_heatmap(
    encoder, decoder, 
    sentence=sample_sentence_bn, 
    en_word2index=en_word2index, 
    de_index2word=de_index2word, 
    lang_code="bengali", 
    device=device
)

print("Attention heatmaps generated for Bengali!")

# ENGLISH-HINDI TRANSLATION

## Data Loading

In [ ]:
source_sentences_train = []
target_sentences_train = []
source_sentences_test = []
id_train = []
id_test = []

with open(train_path, 'r') as file:
    data = json.load(file)

for language_pair, language_data in data.items():
    if language_pair == "English-Hindi":
        print(f"Language Pair: {language_pair}")
        for data_type, data_entries in language_data.items():
            print(f"  Data Type: {data_type}")
            for entry_id, entry_data in data_entries.items():
                source = entry_data["source"]
                if data_type == "Test":
                    source_sentences_test.append(source)
                    id_test.append(entry_id)
                else:
                    target = entry_data["target"]
                    source_sentences_train.append(source)
                    target_sentences_train.append(target)
                    id_train.append(entry_id)

with open(test_path, 'r') as file:
    data = json.load(file)

for language_pair, language_data in data.items():
    if language_pair == "English-Hindi":
        print(f"Language Pair: {language_pair}")
        for data_type, data_entries in language_data.items():
            print(f"  Data Type: {data_type}")
            for entry_id, entry_data in data_entries.items():
                source = entry_data["source"]
                if data_type == "Test":
                    source_sentences_test.append(source)
                    id_test.append(entry_id)
                else:
                    target = entry_data["target"]
                    source_sentences_train.append(source)
                    target_sentences_train.append(target)
                    id_train.append(entry_id)

print(f"Training samples: {len(source_sentences_train)}")
print(f"Test samples: {len(source_sentences_test)}")

## Data Preprocessing

In [ ]:
target_sentences_train = [re.sub(r'[a-zA-Z]','',hi) for hi in target_sentences_train] #optional

english_tokens = preprocess_english(source_sentences_train)
english_test = preprocess_english(source_sentences_test)
hindi_tokens = preprocess_hindi(target_sentences_train)

en_train = english_tokens
en_test = english_test
de_train = hindi_tokens

## VISUALIZE CORPORA (English - Hindi)

In [ ]:
plot_token_distribution(english_tokens, "English Corpus")
plot_token_distribution(hindi_tokens, "Hindi Corpus")

plot_sentence_lengths(english_tokens, "English")
plot_sentence_lengths(hindi_tokens, "Hindi")

plot_zipf_distribution(english_tokens, "English Corpus")
plot_zipf_distribution(hindi_tokens, "Hindi Corpus")

fit_zipf_slope(english_tokens, "English Corpus")
fit_zipf_slope(hindi_tokens, "Hindi Corpus")

## Build vocabulary

In [ ]:
en_index2word = ["<PAD>", "<SOS>", "<EOS>"]
de_index2word = ["<PAD>", "<SOS>", "<EOS>"]

for ds in [en_train, en_test]:
    for sent in ds:
        for token in sent:
            if token not in en_index2word:
                en_index2word.append(token)

for ds in [de_train]:
    for sent in ds:
        for token in sent:
            if token not in de_index2word:
                de_index2word.append(token)

en_word2index = {token: idx for idx, token in enumerate(en_index2word)}
de_word2index = {token: idx for idx, token in enumerate(de_index2word)}

print(f"English vocabulary size: {len(en_word2index)}")
print(f"Bengali vocabulary size: {len(de_word2index)}")

## Encode and pad sequences

In [ ]:
en_train_encoded = [encode_and_pad(en_word2index, sent, config.SEQ_LENGTH) for sent in en_train]
en_test_encoded = [encode_and_pad(en_word2index, sent, config.SEQ_LENGTH) for sent in en_test]
de_train_encoded = [encode_and_pad(de_word2index, sent, config.SEQ_LENGTH) for sent in de_train]

## Prepare DataLoader

In [ ]:
train_x = np.array(en_train_encoded)
train_y = np.array(de_train_encoded)
test_x = np.array(en_test_encoded)

train_lens = torch.tensor([len(seq) - seq.count(0) for seq in train_x.tolist()], dtype=torch.long)

train_ds = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y), train_lens)
train_dl = DataLoader(train_ds, shuffle=True, batch_size=config.BATCH_SIZE, drop_last=True)

test_ds = TensorDataset(torch.from_numpy(test_x))
test_dl = DataLoader(test_ds, batch_size=config.BATCH_SIZE)

print(f"Training DataLoader: {len(train_dl)} batches")
print(f"Test DataLoader: {len(test_dl)} batches")

## Initialize models

In [ ]:
encoder = Encoder(len(en_word2index), config.EMBEDDING_DIM, config.UNITS, config.BATCH_SIZE, config.DROPOUT).to(device)
decoder = Decoder(len(de_word2index), config.EMBEDDING_DIM, config.UNITS, config.UNITS, config.BATCH_SIZE, config.DROPOUT).to(device)

print("Models initialized successfully!")
print(f"Encoder parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Decoder parameters: {sum(p.numel() for p in decoder.parameters()):,}")

## Train the model

In [ ]:
print("Starting training...")
train_losses = train_model(encoder, decoder, train_dl, en_word2index, de_word2index, config, device)

## Plotting the training and validation loss curves

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have a list `train_losses` storing loss per epoch
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker='o')
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.tight_layout()
plt.savefig("training_loss_curve_2.png", dpi=300)
plt.close()

print("Saved training loss curve as training_loss_curve_2.png")


## Save predictions with beam search

In [ ]:
print("Generating predictions with beam search...")
save_predictions_with_beam_search(
    encoder=encoder,
    decoder=decoder,
    test_dl=test_dl,
    test_ids=id_test,
    max_len=config.SEQ_LENGTH,
    sos_idx=en_word2index["<SOS>"],
    eos_idx=en_word2index["<EOS>"],
    de_index2word=de_index2word,
    output_file="answersH.csv",
    device=device,
    beam_width=config.BEAM_WIDTH
)

print("Training and prediction completed successfully!")

## ATTENTION HEATMAP VISUALIZATION

In [ ]:

sample_sentence_hi = "He completed his degree at the university."
# Generate Hindi attention heatmap
generate_attention_heatmap(
    encoder, decoder, 
    sentence=sample_sentence_hi, 
    en_word2index=en_word2index, 
    de_index2word=de_index2word, 
    lang_code="hindi", 
    device=device
)

print("Attention heatmaps generated for Hindi!")

# FINAL OUTPUT

## Input CSV Files

In [ ]:
df1 = pd.read_csv("answersB.csv") # Bengali
df2 = pd.read_csv("answersH.csv")  # Hindi

## Merge CSV Files

In [ ]:
df3 = pd.concat([df1, df2]) #Concat
df3

In [ ]:
df3.to_csv('answersBH.csv', index = False)

## Output Final CSV File

In [ ]:
filtered_data = pd.read_csv("answersBH.csv")

In [ ]:
answer = "answer.csv"
with open(answer, "w") as f:
  f.writelines("ID\tTranslation\n")
  for i in range(filtered_data.shape[0]):
    f.writelines(f'{filtered_data["ID"][i]}\t"{filtered_data["Translation"][i]}"\n')

#  MODEL ARCHITECTURE VISUALIZATION

In [ ]:
# Create dummy inputs for visualization
sample_input = torch.randint(0, 100, (config.BATCH_SIZE, config.SEQ_LENGTH)).to(device)
sample_len = torch.full((config.BATCH_SIZE,), config.SEQ_LENGTH, dtype=torch.long).to(device)

# Initialize models
encoder = Encoder(len(en_word2index), config.EMBEDDING_DIM, config.UNITS, config.BATCH_SIZE, config.DROPOUT).to(device)
decoder = Decoder(len(de_word2index), config.EMBEDDING_DIM, config.UNITS, config.UNITS, config.BATCH_SIZE, config.DROPOUT).to(device)

# Forward pass for visualization
enc_output, enc_hidden = encoder(sample_input, sample_len, device)
dec_input = torch.randint(0, len(de_word2index), (config.BATCH_SIZE, 1)).to(device)
dec_output, dec_hidden, attn = decoder(dec_input, enc_hidden, enc_output)

# Combine encoder and decoder graph
combined = make_dot(
    dec_output,
    params=dict(
        list(encoder.named_parameters()) + list(decoder.named_parameters())
    ),
    show_attrs=True,
    show_saved=True
)

# Save as PNG and PDF
os.makedirs("visualizations", exist_ok=True)
combined.render("visualizations/model_architecture_gru_attention", format="png")
print("Model architecture diagram saved as 'visualizations/model_architecture_gru_attention.png'")
